# Map ESCO to ONET for Job Zones (Education)

Merge O*NET crosswalk data with job zones to enable mapping ESCO occupations to O*NET education levels. The job zones in O*NET indicate education and experience requirements.

**Note:** This notebook explores the O*NET mapping process. Future implementation will move reusable logic to `src/`.

This notebook contains the exploration and can be used for testing individual projects.

## 0. Setup

### 0.01 Import Required Libraries

In [1]:
import pandas as pd
from pathlib import Path

# Import our config
import sys
sys.path.append(str(Path.cwd().parent))
from src.config import load_config

### 0.02 Load Configuration

In [2]:
# Get project root
project_root = Path.cwd().parent

# Load project config
config = load_config()

print("✓ Configuration loaded")

✓ Configuration loaded


### 0.03 Set Up Paths

In [3]:
# Set up paths for O*NET data files
onet_dir = project_root / "data" / "bronze" / "onet"
crosswalk_file = onet_dir / "esco_onet_crosswalk.csv"
job_zones_file = onet_dir / "onet_job_zones.txt"

print(f"✓ Crosswalk file: {crosswalk_file}")
print(f"✓ Job zones file: {job_zones_file}")

✓ Crosswalk file: /Users/lauren/repos/PAD2Skills/data/bronze/onet/esco_onet_crosswalk.csv
✓ Job zones file: /Users/lauren/repos/PAD2Skills/data/bronze/onet/onet_job_zones.txt


## 1. Load and Merge O*NET Data

### 1.01 Load ESCO-ONET Crosswalk

In [4]:
# Load ESCO-ONET crosswalk
crosswalk_df = pd.read_csv(crosswalk_file)

print(f"✓ Loaded {len(crosswalk_df)} crosswalk records")
print(f"  Columns: {', '.join(crosswalk_df.columns)}")
print(f"\nFirst few rows:")
crosswalk_df.head()

✓ Loaded 4253 crosswalk records
  Columns: O*NET Id, O*NET Title, O*NET Description, ESCO or ISCO URI, ESCO or ISCO Title, ESCO or ISCO Description, Type of Match

First few rows:


,O*NET Id,O*NET Title,O*NET Description,ESCO or ISCO URI,ESCO or ISCO Title,ESCO or ISCO Description,Type of Match
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/5c5b153e...,secretary general,Secretaries general head international governm...,closeMatch
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/6c3fd65e...,chief executive officer,Chief executive officers hold the highest rank...,exactMatch
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/c64a6e4e...,chief operating officer,Chief operating officers are the right hand an...,broadMatch
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/4be4ea31...,airport chief executive,Airport chief executives lead a group of airpo...,broadMatch
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/73b10c97...,social entrepreneur,Social entrepreneurs create innovative product...,broadMatch


### 1.02 Load O*NET Job Zones

In [5]:
# Load O*NET job zones (tab-separated file)
job_zones_df = pd.read_csv(job_zones_file, sep='\t')

print(f"✓ Loaded {len(job_zones_df)} job zone records")
print(f"  Columns: {', '.join(job_zones_df.columns)}")
print(f"\nFirst few rows:")
job_zones_df.head()

✓ Loaded 923 job zone records
  Columns: O*NET-SOC Code, Job Zone, Date, Domain Source

First few rows:


,O*NET-SOC Code,Job Zone,Date,Domain Source
0,11-1011.00,5,08/2023,Analyst
1,11-1011.03,5,08/2021,Analyst
2,11-1021.00,4,08/2023,Analyst
3,11-1031.00,4,06/2008,Analyst
4,11-2011.00,4,08/2018,Analyst


### 1.03 Merge Crosswalk with Job Zones

In [6]:
# Merge on O*NET ID
# The crosswalk uses "O*NET Id" and job zones uses "O*NET-SOC Code"
onet_merged_df = crosswalk_df.merge(
    job_zones_df,
    left_on='O*NET Id',
    right_on='O*NET-SOC Code',
    how='left'
)

print(f"✓ Merged {len(onet_merged_df)} records")
print(f"  Original crosswalk: {len(crosswalk_df)} records")
print(f"  Job zones: {len(job_zones_df)} records")
print(f"\nMerged columns: {', '.join(onet_merged_df.columns)}")
print(f"\nFirst few rows:")
onet_merged_df.head()

✓ Merged 4253 records
  Original crosswalk: 4253 records
  Job zones: 923 records

Merged columns: O*NET Id, O*NET Title, O*NET Description, ESCO or ISCO URI, ESCO or ISCO Title, ESCO or ISCO Description, Type of Match, O*NET-SOC Code, Job Zone, Date, Domain Source

First few rows:


,O*NET Id,O*NET Title,O*NET Description,ESCO or ISCO URI,ESCO or ISCO Title,ESCO or ISCO Description,Type of Match,O*NET-SOC Code,Job Zone,Date,Domain Source
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/5c5b153e...,secretary general,Secretaries general head international governm...,closeMatch,11-1011.00,5.0,08/2023,Analyst
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/6c3fd65e...,chief executive officer,Chief executive officers hold the highest rank...,exactMatch,11-1011.00,5.0,08/2023,Analyst
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/c64a6e4e...,chief operating officer,Chief operating officers are the right hand an...,broadMatch,11-1011.00,5.0,08/2023,Analyst
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/4be4ea31...,airport chief executive,Airport chief executives lead a group of airpo...,broadMatch,11-1011.00,5.0,08/2023,Analyst
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/73b10c97...,social entrepreneur,Social entrepreneurs create innovative product...,broadMatch,11-1011.00,5.0,08/2023,Analyst


### 1.04 Clean and Rename Columns

In [7]:
# Rename columns
onet_merged_df = onet_merged_df.rename(columns={
    'O*NET Id': 'onet_id',
    'O*NET Title': 'onet_title',
    'O*NET Description': 'onet_description',
    'ESCO or ISCO URI': 'uri',
    'Job Zone': 'job_zone',
    'ESCO or ISCO Title': 'esco_title',
    'ESCO or ISCO Description': 'esco_description'
})

# Drop unnecessary columns
columns_to_drop = [
    'Type of Match',
    'O*NET-SOC Code',
    'Date',
    'Domain Source'
]
onet_merged_df = onet_merged_df.drop(columns=columns_to_drop)

print(f"✓ Cleaned dataframe")
print(f"  Remaining columns: {', '.join(onet_merged_df.columns)}")
print(f"\nFirst few rows:")
onet_merged_df.head()

✓ Cleaned dataframe
  Remaining columns: onet_id, onet_title, onet_description, uri, esco_title, esco_description, job_zone

First few rows:


,onet_id,onet_title,onet_description,uri,esco_title,esco_description,job_zone
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/5c5b153e...,secretary general,Secretaries general head international governm...,5.0
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/6c3fd65e...,chief executive officer,Chief executive officers hold the highest rank...,5.0
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/c64a6e4e...,chief operating officer,Chief operating officers are the right hand an...,5.0
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/4be4ea31...,airport chief executive,Airport chief executives lead a group of airpo...,5.0
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/73b10c97...,social entrepreneur,Social entrepreneurs create innovative product...,5.0


### 1.05 Extract ESCO ID from URI

In [8]:
# Extract ESCO ID from the URI (last part after the last slash)
onet_merged_df['esco_id'] = onet_merged_df['uri'].str.split('/').str[-1]

# Drop the URI column
onet_merged_df = onet_merged_df.drop(columns=['uri'])

print(f"✓ Extracted ESCO ID")
print(f"  Remaining columns: {', '.join(onet_merged_df.columns)}")
print(f"\nFirst few rows:")
onet_merged_df.head()

✓ Extracted ESCO ID
  Remaining columns: onet_id, onet_title, onet_description, esco_title, esco_description, job_zone, esco_id

First few rows:


,onet_id,onet_title,onet_description,esco_title,esco_description,job_zone,esco_id
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,secretary general,Secretaries general head international governm...,5.0,5c5b153e-4bf8-4f3d-973d-12fabf306d12
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,chief executive officer,Chief executive officers hold the highest rank...,5.0,6c3fd65e-2d24-47d8-bc22-9e93512bdcc2
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,chief operating officer,Chief operating officers are the right hand an...,5.0,c64a6e4e-5b38-4f93-b26d-aded817aeaf3
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,airport chief executive,Airport chief executives lead a group of airpo...,5.0,4be4ea31-1211-4f0c-82bb-f6fe10791f4d
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,social entrepreneur,Social entrepreneurs create innovative product...,5.0,73b10c97-b003-45dc-a36d-c1d585c04be1


## 2. Merge O*NET onto ESCO Dataset

### 2.01 Load Prepared ESCO Occupations

In [9]:
# Load the prepared ESCO occupations
clean_esco_dir = project_root / "data" / "silver" / "clean_esco"
esco_prepared_file = clean_esco_dir / "esco_occupations_prepared.csv"

df_occupations = pd.read_csv(esco_prepared_file)

print(f"Loaded {len(df_occupations)} ESCO occupations")
print(f"Columns: {list(df_occupations.columns)}")
print(f"\nFirst few rows:")
df_occupations.head()

Loaded 3037 ESCO occupations
Columns: ['esco_id', 'conceptUri', 'preferredLabel', 'description', 'combined_text']

First few rows:


,esco_id,conceptUri,preferredLabel,description,combined_text
0,00030d09-2b3a-4efd-87cc-c4ea39d27c34,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,technical director director of technical arts ...
1,000e93a3-d956-4e45-aacb-f12c83fedf84,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,Metal drawing machine operators set up and ope...,metal drawing machine operator wire drawer for...
2,0019b951-c699-4191-8208-9822882d150c,http://data.europa.eu/esco/occupation/0019b951...,precision device inspector,Precision device inspectors make sure precisio...,precision device inspector precision device qu...
3,0022f466-426c-41a4-ac96-a235c945cf97,http://data.europa.eu/esco/occupation/0022f466...,air traffic safety technician,Air traffic safety technicians provide technic...,air traffic safety technician air traffic safe...
4,002da35b-7808-43f3-83bf-63596b8b351f,http://data.europa.eu/esco/occupation/002da35b...,hospitality revenue manager,Hospitality revenue managers maximise revenue ...,hospitality revenue manager yield manager hosp...


In [10]:
onet_merged_df.columns

Index(['onet_id', 'onet_title', 'onet_description', 'esco_title',
       'esco_description', 'job_zone', 'esco_id'],
      dtype='object')

In [11]:
# Check if esco_id is unique
total_rows = len(onet_merged_df)
unique_esco_ids = onet_merged_df['esco_id'].nunique()
has_duplicates = total_rows != unique_esco_ids

print(f"Total rows: {total_rows}")
print(f"Unique esco_id values: {unique_esco_ids}")
print(f"Has duplicates: {has_duplicates}")

if has_duplicates:
    duplicate_count = total_rows - unique_esco_ids
    print(f"\n⚠️  Found {duplicate_count} duplicate esco_id values")
    
    # Show which esco_ids are duplicated
    duplicated_esco_ids = onet_merged_df[onet_merged_df['esco_id'].duplicated(keep=False)].sort_values('esco_id')
    print(f"\nDuplicated esco_ids ({len(duplicated_esco_ids)} rows):")
    duplicated_esco_ids
else:
    print("\n✓ All esco_id values are unique")

Total rows: 4253
Unique esco_id values: 2652
Has duplicates: True

⚠️  Found 1601 duplicate esco_id values

Duplicated esco_ids (2595 rows):


### 2.02 Merge O*NET Data onto Unique ESCO Occupations

In [12]:
# Prepare onet_merged_df for merge - drop esco_title and esco_description to avoid conflicts
onet_for_merge = onet_merged_df.drop(columns=['esco_title', 'esco_description'])

# Merge O*NET data onto df_occupations by esco_id (left join, allowing duplicates)
df_with_onet = df_occupations.merge(
    onet_for_merge,
    on='esco_id',
    how='left'
)

print(f"✓ Merged O*NET data onto unique ESCO occupations")
print(f"  Original df_occupations rows: {len(df_occupations)}")
print(f"  Merged rows: {len(df_with_onet)} (duplicates expected for multi-mapped ESCO IDs)")
print(f"  Columns: {list(df_with_onet.columns)}")
print(f"\nFirst few rows:")
df_with_onet.head()

✓ Merged O*NET data onto unique ESCO occupations
  Original df_occupations rows: 3037
  Merged rows: 4637 (duplicates expected for multi-mapped ESCO IDs)
  Columns: ['esco_id', 'conceptUri', 'preferredLabel', 'description', 'combined_text', 'onet_id', 'onet_title', 'onet_description', 'job_zone']

First few rows:


,esco_id,conceptUri,preferredLabel,description,combined_text,onet_id,onet_title,onet_description,job_zone
0,00030d09-2b3a-4efd-87cc-c4ea39d27c34,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,technical director director of technical arts ...,27-1011.00,Art Directors,Formulate design concepts and presentation app...,4.0
1,00030d09-2b3a-4efd-87cc-c4ea39d27c34,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,technical director director of technical arts ...,27-2012.05,Media Technical Directors/Managers,Coordinate activities of technical departments...,4.0
2,000e93a3-d956-4e45-aacb-f12c83fedf84,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,Metal drawing machine operators set up and ope...,metal drawing machine operator wire drawer for...,51-4021.00,"Extruding and Drawing Machine Setters, Operato...","Set up, operate, or tend machines to extrude o...",2.0
3,000e93a3-d956-4e45-aacb-f12c83fedf84,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,Metal drawing machine operators set up and ope...,metal drawing machine operator wire drawer for...,51-4022.00,"Forging Machine Setters, Operators, and Tender...","Set up, operate, or tend forging machines to t...",2.0
4,000e93a3-d956-4e45-aacb-f12c83fedf84,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,Metal drawing machine operators set up and ope...,metal drawing machine operator wire drawer for...,51-4023.00,"Rolling Machine Setters, Operators, and Tender...","Set up, operate, or tend machines to roll stee...",2.0


In [13]:
# Analyze occupations without job zones
missing_job_zones = df_with_onet[df_with_onet['job_zone'].isna()]

print(f"Occupations without job zones: {len(missing_job_zones)} rows")
print(f"Unique ESCO IDs without job zones: {missing_job_zones['esco_id'].nunique()}")
print(f"\nBreakdown:")
print(f"  - Total occupations in merged dataset: {len(df_with_onet)}")
print(f"  - Occupations with job zones: {df_with_onet['job_zone'].notna().sum()}")
print(f"  - Occupations without job zones: {df_with_onet['job_zone'].isna().sum()}")

# Check if missing job zones are due to:
# 1. Missing from crosswalk (no onet_id match)
# 2. Missing from job zones file (onet_id exists but no job zone data)
missing_onet_id = missing_job_zones[missing_job_zones['onet_id'].isna()]
has_onet_no_zone = missing_job_zones[missing_job_zones['onet_id'].notna()]

print(f"\nDiagnostics:")
print(f"  - Missing from crosswalk (no onet_id): {len(missing_onet_id)} rows ({missing_onet_id['esco_id'].nunique()} unique ESCO IDs)")
print(f"  - Has onet_id but missing job zone: {len(has_onet_no_zone)} rows ({has_onet_no_zone['esco_id'].nunique()} unique ESCO IDs)")

print(f"\nSample occupations missing from crosswalk:")
missing_onet_id[['esco_id', 'preferredLabel', 'onet_id']].drop_duplicates().head(10)

Occupations without job zones: 587 rows
Unique ESCO IDs without job zones: 584

Breakdown:
  - Total occupations in merged dataset: 4637
  - Occupations with job zones: 4050
  - Occupations without job zones: 587

Diagnostics:
  - Missing from crosswalk (no onet_id): 432 rows (432 unique ESCO IDs)
  - Has onet_id but missing job zone: 155 rows (152 unique ESCO IDs)

Sample occupations missing from crosswalk:


,esco_id,preferredLabel,onet_id
6,0022f466-426c-41a4-ac96-a235c945cf97,air traffic safety technician,NaN
21,0090dd54-057a-4f65-bbe8-f74c4999e8e8,pasta operator,NaN
35,01205776-db8d-4588-a6a6-56d0b9c660a9,housing policy officer,NaN
48,01d74409-5303-4b5c-8925-0cc21c4da8ee,medical writer,NaN
57,02609ece-cea3-43fa-bf38-b6fb821f28b2,tour operator manager,NaN
87,03632d98-0ae3-4dd2-941c-3b48de9a0219,performance production manager,NaN
93,03b55cb5-9735-4582-8e20-2b1cb76635cb,audiology equipment shop manager,NaN
112,046574a1-9bb3-4a80-8a7b-83dc7db1dae2,bricklaying supervisor,NaN
121,04f39bfa-bc03-4480-98bc-b18ce4fe4b4b,accounting manager,NaN
157,066a9d8f-3948-409c-a3cf-ed8ed37cfe22,curing room worker,NaN


### 2.03 Take Minimum Job Zone

In [14]:
# Calculate minimum job zone for each esco_id
# Since df_occupations has unique esco_id values, we group by esco_id from the merged data
df_with_onet_min = df_with_onet.groupby('esco_id').agg({
    'job_zone': 'min'
}).reset_index()

# Rename the job_zone column to job_zone_min
df_with_onet_min = df_with_onet_min.rename(columns={'job_zone': 'job_zone_min'})

# Merge the minimum job zone back to df_occupations
df_final = df_occupations.merge(
    df_with_onet_min,
    on='esco_id',
    how='left'
)

print(f"✓ Calculated minimum job zone and merged back to ESCO occupations")
print(f"  Original df_occupations rows: {len(df_occupations)}")
print(f"  Final rows: {len(df_final)}")
print(f"  Rows match: {len(df_final) == len(df_occupations)}")
print(f"  Columns: {list(df_final.columns)}")
print(f"\nFirst few rows:")
df_final.head()

✓ Calculated minimum job zone and merged back to ESCO occupations
  Original df_occupations rows: 3037
  Final rows: 3037
  Rows match: True
  Columns: ['esco_id', 'conceptUri', 'preferredLabel', 'description', 'combined_text', 'job_zone_min']

First few rows:


,esco_id,conceptUri,preferredLabel,description,combined_text,job_zone_min
0,00030d09-2b3a-4efd-87cc-c4ea39d27c34,http://data.europa.eu/esco/occupation/00030d09...,technical director,Technical directors realise the artistic visio...,technical director director of technical arts ...,4.0
1,000e93a3-d956-4e45-aacb-f12c83fedf84,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,Metal drawing machine operators set up and ope...,metal drawing machine operator wire drawer for...,2.0
2,0019b951-c699-4191-8208-9822882d150c,http://data.europa.eu/esco/occupation/0019b951...,precision device inspector,Precision device inspectors make sure precisio...,precision device inspector precision device qu...,3.0
3,0022f466-426c-41a4-ac96-a235c945cf97,http://data.europa.eu/esco/occupation/0022f466...,air traffic safety technician,Air traffic safety technicians provide technic...,air traffic safety technician air traffic safe...,NaN
4,002da35b-7808-43f3-83bf-63596b8b351f,http://data.europa.eu/esco/occupation/002da35b...,hospitality revenue manager,Hospitality revenue managers maximise revenue ...,hospitality revenue manager yield manager hosp...,4.0


In [15]:
# Fill missing job_zone_min with 9 and convert to int
df_final['job_zone_min'] = df_final['job_zone_min'].fillna(9).astype(int)

print(f"✓ Filled missing job zones with 9 and converted to int")
print(f"  Missing values: {df_final['job_zone_min'].isna().sum()}")
print(f"  Data type: {df_final['job_zone_min'].dtype}")
print(f"\nValue counts:")
print(df_final['job_zone_min'].value_counts().sort_index())

✓ Filled missing job zones with 9 and converted to int
  Missing values: 0
  Data type: int64

Value counts:
job_zone_min
1    111
2    889
3    561
4    711
5    197
9    568
Name: count, dtype: int64


In [16]:
df_final[['preferredLabel', 'job_zone_min']].head(50)

,preferredLabel,job_zone_min
0,technical director,4
1,metal drawing machine operator,2
2,precision device inspector,3
3,air traffic safety technician,9
4,hospitality revenue manager,4
5,medical laboratory assistant,2
6,asphalt laboratory technician,3
7,primary school teaching assistant,3
8,physiotherapist,5
9,performing arts theatre instructor,5


## 3. Label Missing Job Zones with LLM

For ESCO occupations that don't have an O*NET mapping (job_zone_min == 9), use an LLM to assign appropriate job zones based on the occupation description.

### 3.01 Filter Occupations Missing Job Zones

In [17]:
# Filter occupations with missing job zones (job_zone_min == 9)
df_missing = df_final[df_final['job_zone_min'] == 9].copy()

print(f"Occupations missing job zones: {len(df_missing)}")
print(f"Total occupations: {len(df_final)}")
print(f"Percentage missing: {len(df_missing)/len(df_final)*100:.1f}%")

print(f"\nColumns available: {list(df_missing.columns)}")
print(f"\nSample of missing occupations:")
df_missing[['esco_id', 'preferredLabel', 'description']].head(10)

Occupations missing job zones: 568
Total occupations: 3037
Percentage missing: 18.7%

Columns available: ['esco_id', 'conceptUri', 'preferredLabel', 'description', 'combined_text', 'job_zone_min']

Sample of missing occupations:


,esco_id,preferredLabel,description
3,0022f466-426c-41a4-ac96-a235c945cf97,air traffic safety technician,Air traffic safety technicians provide technic...
11,0090dd54-057a-4f65-bbe8-f74c4999e8e8,pasta operator,Pasta operators manufacture dry pasta products...
19,01205776-db8d-4588-a6a6-56d0b9c660a9,housing policy officer,"Housing policy officers research, analyse and ..."
24,01989dd8-36c0-43cc-a82d-2bfc409de9c4,dismantling engineer,Dismantling engineers research and plan the op...
27,01d74409-5303-4b5c-8925-0cc21c4da8ee,medical writer,Medical writers produce and process scientific...
30,021663ca-a367-472e-a1f0-6d8ab5b2d860,legal guardian,Legal guardians legally assist and support min...
34,02609ece-cea3-43fa-bf38-b6fb821f28b2,tour operator manager,Tour operator managers are in charge of managi...
50,03632d98-0ae3-4dd2-941c-3b48de9a0219,performance production manager,Performance production managers take care of a...
54,03b55cb5-9735-4582-8e20-2b1cb76635cb,audiology equipment shop manager,Audiology equipment shop managers assume respo...
61,0458929a-f6c9-40ed-8ce3-caa9f03a6b5b,coffee grinder,Coffee grinders operate grinding machines to g...


### 3.02 Prepare Data for API

The prompt expects JSON format with a `records` array containing objects with `esco_id` and `combined_text` fields.

In [18]:
# Prepare the input data with esco_id and combined_text
# combined_text will include preferredLabel and description
df_missing['combined_text'] = (
    df_missing['preferredLabel'] + '. ' + 
    df_missing['description'].fillna('')
)

# Select only the columns needed for the API
df_api_input = df_missing[['esco_id', 'combined_text']].copy()

print(f"Prepared {len(df_api_input)} occupations for API")
print(f"\nColumns: {list(df_api_input.columns)}")
print(f"\nSample combined_text:")
print(df_api_input['combined_text'].iloc[0][:300])

Prepared 568 occupations for API

Columns: ['esco_id', 'combined_text']

Sample combined_text:
air traffic safety technician. Air traffic safety technicians provide technical support regarding the safety of air traffic control and navigation systems. They design, maintain, install and operate these systems both in the airport and on board the aeroplane according to regulations.


### 3.03 Create Chunks

Split the data into chunks of 50 occupations for API processing.

In [19]:
import numpy as np

# Define chunk size
CHUNK_SIZE = 50

# Split dataframe into chunks
def chunk_dataframe(df, chunk_size):
    """Split a dataframe into chunks of specified size."""
    num_chunks = int(np.ceil(len(df) / chunk_size))
    for i in range(num_chunks):
        yield df.iloc[i * chunk_size:(i + 1) * chunk_size]

# Create list of chunk dataframes
chunks = list(chunk_dataframe(df_api_input, CHUNK_SIZE))

print(f"Total occupations: {len(df_api_input)}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Number of chunks: {len(chunks)}")
print(f"Last chunk size: {len(chunks[-1])}")
print(f"\nFirst chunk preview:")
chunks[0].head()

Total occupations: 568
Chunk size: 50
Number of chunks: 12
Last chunk size: 18

First chunk preview:


,esco_id,combined_text
3,0022f466-426c-41a4-ac96-a235c945cf97,air traffic safety technician. Air traffic saf...
11,0090dd54-057a-4f65-bbe8-f74c4999e8e8,pasta operator. Pasta operators manufacture dr...
19,01205776-db8d-4588-a6a6-56d0b9c660a9,housing policy officer. Housing policy officer...
24,01989dd8-36c0-43cc-a82d-2bfc409de9c4,dismantling engineer. Dismantling engineers re...
27,01d74409-5303-4b5c-8925-0cc21c4da8ee,medical writer. Medical writers produce and pr...


### 3.04 Load Environment and Initialize OpenAI Client

In [20]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
env_path = project_root / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        f"'.env' file not found at {env_path}\n"
        "Please copy .env.example to .env and add your OpenAI API key."
    )

# Load from specific path
load_dotenv(env_path, override=True)

# Get OpenAI API key from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Verify API key is set
if not OPENAI_API_KEY:
    raise ValueError("Missing required environment variable: OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

print("✓ Environment variables loaded")
print(f"  API Key: {OPENAI_API_KEY[:10]}...{OPENAI_API_KEY[-4:]}")
print("✓ OpenAI client initialized")

✓ Environment variables loaded
  API Key: sk-proj-cj...__0A
✓ OpenAI client initialized


### 3.05 Create Function to Call API for Job Zone Labeling

In [21]:
def label_job_zones_with_api(chunk_df, client):
    """
    Call OpenAI API to label job zones for a chunk of occupations.
    
    Args:
        chunk_df: DataFrame with columns esco_id and combined_text
        client: OpenAI client instance
    
    Returns:
        DataFrame with columns esco_id and job_zone
    """
    import json
    
    # Convert chunk to JSON format
    records = chunk_df.to_dict('records')
    input_json = json.dumps({"records": records}, indent=2)
    
    print(f"Calling API...")
    print(f"  Input size: {len(input_json):,} chars")
    print(f"  Occupations: {len(chunk_df)}")
    
    # Call OpenAI API with prompt
    response = client.responses.create(
        prompt={
            "id": "pmpt_695d532bf6c081938296db5067b3d0b6015bedad79cdc6ce",
            "version": "1"
        },
        input=[
            {"role": "user", "content": input_json}
        ],
        reasoning={
            "summary": None
        },
        store=False,
        include=[
            "reasoning.encrypted_content",
            "web_search_call.action.sources"
        ]
    )
    
    # Extract the text from the response
    result_text = None
    for item in response.output:
        if hasattr(item, 'content') and hasattr(item, 'role'):
            result_text = item.content[0].text
            break
    
    if result_text is None:
        raise ValueError("No content found in API response")
    
    # Parse the JSON response
    result_json = json.loads(result_text)
    result_df = pd.DataFrame(result_json['records'])
    
    print(f"  ✓ Response received (response_id: {response.id})")
    print(f"  Output rows: {len(result_df)}")
    
    return result_df

print("✓ Function defined: label_job_zones_with_api()")

✓ Function defined: label_job_zones_with_api()


### 3.06 Process All Chunks

In [22]:
# Process all chunks and collect results
all_labeled_results = []

print(f"Processing {len(chunks)} chunks...")
print("=" * 80)

for i, chunk_df in enumerate(chunks, 1):
    print(f"\nChunk {i}/{len(chunks)}: {len(chunk_df)} occupations")
    
    # Call API to label job zones for this chunk
    result_df = label_job_zones_with_api(chunk_df, client)
    
    # Store the results
    all_labeled_results.append(result_df)

# Combine all results into a single dataframe
df_all_labeled = pd.concat(all_labeled_results, ignore_index=True)

print("\n" + "=" * 80)
print(f"Total labeled rows: {len(df_all_labeled)}")
print(f"Unique occupations labeled: {df_all_labeled['esco_id'].nunique()}")
print(f"Columns: {list(df_all_labeled.columns)}")
print(f"\nJob zone distribution:")
print(df_all_labeled['job_zone'].value_counts().sort_index())

Processing 12 chunks...

Chunk 1/12: 50 occupations
Calling API...
  Input size: 22,166 chars
  Occupations: 50
  ✓ Response received (response_id: resp_01a4f1b72ceeff2e01695d59e81e648197a53fd8f585bae4ad)
  Output rows: 50

Chunk 2/12: 50 occupations
Calling API...
  Input size: 20,926 chars
  Occupations: 50
  ✓ Response received (response_id: resp_0a444aadf3b2ce6001695d5a267b848192a442ffd0f33bc417)
  Output rows: 50

Chunk 3/12: 50 occupations
Calling API...
  Input size: 20,802 chars
  Occupations: 50
  ✓ Response received (response_id: resp_035c9015351b3eac01695d5a6640e4819cbdcde00fb69e8a42)
  Output rows: 50

Chunk 4/12: 50 occupations
Calling API...
  Input size: 21,864 chars
  Occupations: 50
  ✓ Response received (response_id: resp_0c92fb3585f71bbd01695d5a9fb1ac819cacfa36144136373b)
  Output rows: 50

Chunk 5/12: 50 occupations
Calling API...
  Input size: 21,719 chars
  Occupations: 50
  ✓ Response received (response_id: resp_0bbaa50c43eb550801695d5ae0a014819cbc28ba3caf6bca64)

### 3.07 Save Results to CSV

In [23]:
# Set up output directory
output_dir = project_root / "data" / "silver" / "clean_esco"
output_dir.mkdir(parents=True, exist_ok=True)

# Create output filename
output_file = output_dir / "label_missing_onet_job_zone.csv"

# Save to CSV
df_all_labeled.to_csv(output_file, index=False)

print(f"✓ Saved results to: {output_file}")
print(f"  Rows: {len(df_all_labeled):,}")
print(f"  Columns: {len(df_all_labeled.columns)}")
print(f"  Unique occupations: {df_all_labeled['esco_id'].nunique()}")
print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")

✓ Saved results to: /Users/lauren/repos/PAD2Skills/data/silver/clean_esco/label_missing_onet_job_zone.csv
  Rows: 568
  Columns: 2
  Unique occupations: 568
  File size: 21.6 KB


### 3.08 Preview Labeled Results

In [24]:
# Merge labeled results back to get occupation names for preview
df_labeled_preview = df_all_labeled.merge(
    df_missing[['esco_id', 'preferredLabel']],
    on='esco_id',
    how='left'
)

print("Sample of labeled occupations:")
print(f"\nJob Zone 1 (Little or No Preparation):")
print(df_labeled_preview[df_labeled_preview['job_zone'] == 1][['preferredLabel', 'job_zone']].head(5))

print(f"\nJob Zone 2 (Some Preparation):")
print(df_labeled_preview[df_labeled_preview['job_zone'] == 2][['preferredLabel', 'job_zone']].head(5))

print(f"\nJob Zone 3 (Medium Preparation):")
print(df_labeled_preview[df_labeled_preview['job_zone'] == 3][['preferredLabel', 'job_zone']].head(5))

print(f"\nJob Zone 4 (Considerable Preparation):")
print(df_labeled_preview[df_labeled_preview['job_zone'] == 4][['preferredLabel', 'job_zone']].head(5))

print(f"\nJob Zone 5 (Extensive Preparation):")
print(df_labeled_preview[df_labeled_preview['job_zone'] == 5][['preferredLabel', 'job_zone']].head(5))

Sample of labeled occupations:

Job Zone 1 (Little or No Preparation):
           preferredLabel  job_zone
45   linen room attendant         1
59               stand-in         1
66       volunteer mentor         1
119                medium         1
246            astrologer         1

Job Zone 2 (Some Preparation):
               preferredLabel  job_zone
1              pasta operator         2
5              legal guardian         2
9              coffee grinder         2
14  spinning textile operator         2
15         curing room worker         2

Job Zone 3 (Medium Preparation):
                   preferredLabel  job_zone
0   air traffic safety technician         3
10         bricklaying supervisor         3
11  marine engineering technician         3
13            maritime instructor         3
18                puppet designer         3

Job Zone 4 (Considerable Preparation):
                   preferredLabel  job_zone
2          housing policy officer         4
3            di